In [20]:
## This notebook investigates the association between individual diabetes medications administered during hospital stays and the likelihood of patient readmission.

## The goal is to identify specific medications that may be linked to higher or lower readmission rates. 
## Understanding these associations may help highlight treatment patterns correlated with readmission risk.

## Because this dataset is observational, the analysis focuses on statistical associations rather than causal relationships. 
## Logistic regression models will be used to estimate the relationship between medication use and readmission while controlling for patient characteristics 
## such as age, hospital utilization, and diagnostic complexity.

##Medication use in hospital settings is not randomly assigned. 
##Patients receiving certain treatments may differ systematically in disease severity or clinical presentation, a phenomenon known as confounding by indication. 
##Therefore, observed associations between medication use and readmission may reflect underlying patient characteristics rather than direct effects of the medications themselves.

import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\palla\OneDrive\Documents\Coding Projects\Hospital Diabetes Dataset\diabetic_data_utf8.csv")

df.shape

df.head()

df.columns

Index(['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight',
       'admission_type_id', 'discharge_disposition_id', 'admission_source_id',
       'time_in_hospital', 'payer_code', 'medical_specialty',
       'num_lab_procedures', 'num_procedures', 'num_medications',
       'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1',
       'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

In [22]:
#Checking columns with missing values

df.isnull().sum().sort_values(ascending=False)

max_glu_serum               96420
A1Cresult                   84748
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
encounter_id                    0
patient_nbr                     0
number_inpatient                0
number_emergency                0
diag_1                          0
diag_2                          0
number_diagnoses                0
diag_3                          0
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [23]:
# Proportion of patients readmitted within 30 days, after 30 days, and never readmitted

readmitted = df["readmitted"].value_counts(normalize=True)
print(readmitted)

readmitted
NO     0.539119
>30    0.349282
<30    0.111599
Name: proportion, dtype: float64


In [24]:
# Create a binary target variable for readmission within 30 days


medication_cols = [
    "metformin","repaglinide","nateglinide","chlorpropamide","glimepiride",
    "acetohexamide","glipizide","glyburide","tolbutamide","pioglitazone",
    "rosiglitazone","acarbose","miglitol","troglitazone","tolazamide",
    "examide","citoglipton","insulin","glyburide-metformin",
    "glipizide-metformin","glimepiride-pioglitazone",
    "metformin-rosiglitazone","metformin-pioglitazone"
]

med_usage_counts = (df[medication_cols] != "No").sum()

med_usage_counts.sort_values(ascending=False)

selected_meds = med_usage_counts[med_usage_counts >= 500].index.tolist()

print("Selected medications for analysis:")
print(selected_meds)

Selected medications for analysis:
['metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'glyburide-metformin']


In [25]:
#create a new column for each medication indicating whether it was used (1) or not (0)

used_columns= []

for med in selected_meds:
    new_col = med + "_used"
    df[new_col] = (df[med] != "No").astype(int)
    used_columns.append(new_col)

df[used_columns].head()


,metformin_used,repaglinide_used,nateglinide_used,glimepiride_used,glipizide_used,glyburide_used,pioglitazone_used,rosiglitazone_used,insulin_used,glyburide-metformin_used
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,1,0
2,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,1,0,0,0,1,0


In [28]:
df[used_cols].sum().sort_values(ascending=False)

NameError: name 'used_cols' is not defined

In [ ]:
# Check the proportion of patients readmitted within 30 days in the dataset

df["readmit_binary"] = (df["readmitted"] == "<30").astype(int)

df["readmit_binary"].mean()

np.float64(0.11159915885462728)

In [ ]:
#create a dictionary to hold readmission rates for each medication

readmit_rates = {}

for col in used_cols:
    
    rate = df.groupby(col)["readmit_binary"].mean()
    
    readmit_rates[col] = rate[1]   # rate among drug users

pd.Series(readmit_rates).sort_values(ascending=False)

repaglinide_used            0.133203
insulin_used                0.121380
glipizide_used              0.114457
nateglinide_used            0.113798
glyburide-metformin_used    0.110482
glyburide_used              0.106291
pioglitazone_used           0.105622
rosiglitazone_used          0.104478
glimepiride_used            0.102100
metformin_used              0.097008
dtype: float64

In [ ]:
severity_cols = [
    "age",
    "time_in_hospital",
    "num_medications",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
    "number_diagnoses"
]

df[severity_cols].describe().round(2)

,time_in_hospital,num_medications,number_inpatient,number_emergency,number_outpatient,number_diagnoses
count,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00
mean,4.40,16.02,0.64,0.20,0.37,7.42
std,2.99,8.13,1.26,0.93,1.27,1.93
min,1.00,1.00,0.00,0.00,0.00,1.00
25%,2.00,10.00,0.00,0.00,0.00,6.00
50%,4.00,15.00,0.00,0.00,0.00,8.00
75%,6.00,20.00,1.00,0.00,0.00,9.00
max,14.00,81.00,21.00,76.00,42.00,16.00


In [ ]:
feature_cols = used_cols + severity_cols

X = df[feature_cols]
y = df["readmit_binary"]

X.shape



print(X.shape)   # 17 predictors
print(y.shape)      #one outcome per patient

(101766, 17)
(101766,)


In [19]:
from sklearn.model_selection import train_test_split

X_encoded = pd.get_dummies(X, drop_first=True)
X_encoded.shape

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape #confirming the training set has 80% of the data and 17 predictors


NameError: name 'X' is not defined

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
coefficients = model.coef_[0]

coef_table = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": coefficients
})

print(coef_table)

NameError: name 'model' is not defined

In [ ]:
coef_table["odds_ratio"] = np.exp(coef_table["coefficient"])

coef_table.sort_values("odds_ratio", ascending=False)

NameError: name 'coef_table' is not defined

In [ ]:
coef_table[coef_table["feature"].str.contains("_used")]\
.sort_values("odds_ratio", ascending=False)

NameError: name 'coef_table' is not defined